In [ ]:
# ═══════════════════════════════════════════════════════════════
# LUMISENSE GRID — DATA POPULATION
# ═══════════════════════════════════════════════════════════════

from snowflake.snowpark.context import get_active_session
import random
from datetime import datetime, timedelta

session = get_active_session()
print("✅ Session ready:", session.get_current_database())

In [ ]:
# ─── ZONES ────────────────────────────────────────────────────
ZONES = {
    "residential": {
        "lights": ["L1","L2","L3","L4","L5"],
        "offset": 0,
        "description": "Residential streets — moderate light"
    },
    "main_road": {
        "lights": ["L6","L7","L8","L9","L10"],
        "offset": 50,
        "description": "Main roads — open sky, brighter"
    },
    "park": {
        "lights": ["L11","L12","L13","L14","L15"],
        "offset": 100,
        "description": "Park area — most open, brightest"
    },
    "industrial": {
        "lights": ["L16","L17","L18","L19","L20"],
        "offset": -80,
        "description": "Industrial zone — covered, darker"
    },
}

LIGHT_VARIATION = {f"L{i}": random.randint(-30, 30) for i in range(1, 21)}

# ─── HELPERS ──────────────────────────────────────────────────
def get_zone(light_id):
    for zone, data in ZONES.items():
        if light_id in data["lights"]:
            return zone
    return "residential"

def get_ldr(hour, light_id):
    zone   = get_zone(light_id)
    offset = ZONES[zone]["offset"] + LIGHT_VARIATION[light_id]
    if   0 <= hour <= 5:  base = random.randint(30,  150)
    elif 6 <= hour <= 8:  base = random.randint(200, 500)
    elif 9 <= hour <= 17: base = random.randint(600, 950)
    elif 18 <= hour <= 21:base = random.randint(250, 550)
    else:                 base = random.randint(80,  250)
    return max(0, min(1023, base + offset))

In [ ]:
# ─── POPULATE LDR_GRID_DATA ───────────────────────────────────
print("\n📡 Populating LDR_GRID_DATA...")
print("   3 days × 20 lights × every 90s = ~19,200 rows")

lights       = [f"L{i}" for i in range(1, 21)]
rows         = []
now          = datetime.now()
three_days_ago = now - timedelta(days=3)
current_time = three_days_ago

while current_time <= now:
    hour = current_time.hour
    for light_id in lights:
        ldr = get_ldr(hour, light_id)
        rows.append((light_id, ldr, hour,
                     current_time.strftime('%Y-%m-%d %H:%M:%S')))
    current_time += timedelta(seconds=90)

print(f"   → {len(rows):,} rows generated")

# Batch insert
chunk_size = 1000
inserted   = 0
for i in range(0, len(rows), chunk_size):
    chunk = rows[i:i + chunk_size]
    for row in chunk:
        session.sql("""
            INSERT INTO LUMISENSE_DB.LUMISENSE_SCHEMA.LDR_GRID_DATA
                (light_id, ldr_value, hour_of_day, timestamp)
            VALUES (?, ?, ?, ?)
        """, params=list(row)).collect()
    inserted += len(chunk)
    print(f"   → {inserted:,}/{len(rows):,} inserted...")

print(f"✅ LDR_GRID_DATA done! ({len(rows):,} rows)")

# ─── POPULATE LDR_LED_OPTIMIZATION_LOG ────────────────────────
print("\n🧠 Populating LDR_LED_OPTIMIZATION_LOG...")

opt_rows = [
    (6,  20, 15, 80, 85,
     "avg_ldr 580 near ldr_max 599 — safe to dim",
     (now - timedelta(days=2)).strftime('%Y-%m-%d %H:%M:%S')),

    (8,  30, 25, 70, 75,
     "avg_ldr 570 consistently high — reducing brightness",
     (now - timedelta(days=2)).strftime('%Y-%m-%d %H:%M:%S')),

    (13, 5,  5,  95, 95,
     "already at minimum — no change applied",
     (now - timedelta(days=1)).strftime('%Y-%m-%d %H:%M:%S')),

    (11, 10, 8,  90, 92,
     "midnight readings show LDR 600+ stable — dimming safely",
     (now - timedelta(hours=12)).strftime('%Y-%m-%d %H:%M:%S')),

    (15, 15, 12, 85, 88,
     "late night LDR near max — small brightness reduction",
     (now - timedelta(hours=6)).strftime('%Y-%m-%d %H:%M:%S')),
]

for row in opt_rows:
    session.sql("""
        INSERT INTO LUMISENSE_DB.LUMISENSE_SCHEMA.LDR_LED_OPTIMIZATION_LOG
            (scenario_id, old_brightness, new_brightness,
             old_saving, new_saving, reason, timestamp)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, params=list(row)).collect()

print(f"✅ LDR_LED_OPTIMIZATION_LOG done! ({len(opt_rows)} rows)")

# ─── VERIFY ALL TABLES ────────────────────────────────────────
print("\n📊 Verifying data...")

tables = [
    "LDR_GRID_DATA",
    "LDR_LED_SCENARIO_RULES",
    "LDR_LED_OPTIMIZATION_LOG",
]

for table in tables:
    result = session.sql(f"""
        SELECT COUNT(*) AS cnt
        FROM LUMISENSE_DB.LUMISENSE_SCHEMA.{table}
    """).collect()[0]
    print(f"   {table:<35} → {result['CNT']:>8,} rows")

print("\n🎉 All tables populated successfully!")